In [2]:
import os
import librosa
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt 

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

2025-11-14 12:21:18.315284: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-14 12:21:18.322157: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-14 12:21:18.565738: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-14 12:21:19.860751: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or

##  Main funkction

In [3]:
class DataPrep:

    def __init__(self,
                 dataset_path,
                 segment_duration=5.0,
                 sr=44100,
                 hop_length=1024,
                 fmin=32.7,
                 n_bins=360,
                 bins_per_octave=60):

        self.dataset_path = dataset_path
        self.segment_duration = segment_duration
        self.sr = sr
        self.hop_length = hop_length
        self.fmin = fmin
        self.n_bins = n_bins
        self.bins_per_octave = bins_per_octave

        # listy na dane
        self.X_list = []
        self.Y_list = []

    # -------------------------
    #   HELPER: CQT
    # -------------------------
    def seg_to_cqt(self, y):
        C = np.abs(librosa.cqt(
            y=y,
            sr=self.sr,
            hop_length=self.hop_length,
            fmin=self.fmin,
            n_bins=self.n_bins,
            bins_per_octave=self.bins_per_octave
        ))
        C_db = librosa.amplitude_to_db(C, ref=np.max)
        return C_db

    # -------------------------
    #   HELPER: normalizacja segmentu
    # -------------------------
    @staticmethod
    def normalize_segment(y):
        return y / (np.max(np.abs(y)) + 1e-9)

    # -------------------------
    #   GŁÓWNA METODA
    # -------------------------
    def build(self):

        print(tf.__version__) 
        for split in ["train", "test"]:  
            split_path = os.path.join(self.dataset_path, split)

            for track_name in os.listdir(split_path):
                track_path = os.path.join(split_path, track_name)

                # musi być katalog z utworem
                if not os.path.isdir(track_path):
                    continue

                # lista plików w utworze
                files = os.listdir(track_path)

                # sprawdzamy czy są stem’y
                if "mixture.wav" not in files:
                    continue

                mix_path    = os.path.join(track_path, "mixture.wav")
                bass_path   = os.path.join(track_path, "bass.wav")
                drums_path  = os.path.join(track_path, "drums.wav")
                other_path  = os.path.join(track_path, "other.wav")
                vocals_path = os.path.join(track_path, "vocals.wav")

                print("Przetwarzam:", track_path)

        # --- dalej Twoje segmentowanie i CQT ---
                # load
                mix, sr = librosa.load(mix_path, sr=self.sr, mono=True)
                bass,_   = librosa.load(bass_path,   sr=self.sr, mono=True)
                drums,_  = librosa.load(drums_path,  sr=self.sr, mono=True)
                other,_  = librosa.load(other_path,  sr=self.sr, mono=True)
                vocals,_ = librosa.load(vocals_path, sr=self.sr, mono=True)

                # segmentation
                segment_samples = int(self.segment_duration * self.sr)
                num_segments = len(mix) // segment_samples

                for seg_idx in range(num_segments):

                    start = seg_idx * segment_samples
                    stop = start + segment_samples

                    # waveform segments
                    mix_seg    = self.normalize_segment(mix[start:stop])
                    bass_seg   = self.normalize_segment(bass[start:stop])
                    drums_seg  = self.normalize_segment(drums[start:stop])
                    other_seg  = self.normalize_segment(other[start:stop])
                    vocals_seg = self.normalize_segment(vocals[start:stop])

                    # CQT for each segment
                    mix_cqt    = self.seg_to_cqt(mix_seg)
                    bass_cqt   = self.seg_to_cqt(bass_seg)
                    drums_cqt  = self.seg_to_cqt(drums_seg)
                    other_cqt  = self.seg_to_cqt(other_seg)
                    vocals_cqt = self.seg_to_cqt(vocals_seg)

                    # shape (n_bins, T, 1)
                    X_seg = mix_cqt[..., np.newaxis]

                    # shape (n_bins, T, 4)
                    Y_seg = np.stack(
                        [bass_cqt, drums_cqt, other_cqt, vocals_cqt],
                        axis=-1
                    )

                    self.X_list.append(X_seg)
                    self.Y_list.append(Y_seg)

        # stack all
        X = np.stack(self.X_list, axis=0)
        Y = np.stack(self.Y_list, axis=0)

        # convert to TF tensors
        X_tf = tf.convert_to_tensor(X, dtype=tf.float32)
        Y_tf = tf.convert_to_tensor(Y, dtype=tf.float32)

        return X_tf, Y_tf



    def save_data(self, path="/home/ciona/projects/RCOLM/data/raw_data/musdb18/dataset"):
        X, Y = self.build()
        np.savez_compressed(path, X=X, Y=Y)
        print("Zapisano dataset:", path)
        

dp = DataPrep(dataset_path = "/home/ciona/projects/RCOLM/data/raw_data/musdb18")

dp.save_data()

2.20.0
Przetwarzam: /home/ciona/projects/RCOLM/data/raw_data/musdb18/train/Secret Mountains - High Horse


KeyboardInterrupt: 